In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from transformers_sae import _autoreload
from transformers_sae.ops import MemoryTrackingMode
from transformers_sae.replacement_model import GemmaReplacement, make_replacement_model

# Tweak TRAINING_BATCH_SIZE for your hardware if necessary
if torch.cuda.is_available():
    TRAINING_DEVICE = "cuda:0"
    TRAINING_BATCH_SIZE = 2
elif torch.mps.is_available():
    TRAINING_DEVICE = "mps:0"
    TRAINING_BATCH_SIZE = 2
else:
    TRAINING_DEVICE = "cpu"
    TRAINING_BATCH_SIZE = 2

model_id = "google/gemma-2-2b"
tokenizer = AutoTokenizer.from_pretrained(model_id)

with MemoryTrackingMode() as mtm:
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map=TRAINING_DEVICE,
        dtype=torch.bfloat16,
        use_safetensors=True,
    )
    model = make_replacement_model(
        model,
        {},
        num_layers=model.config.num_hidden_layers,
        context_length=1024,  # model.config.max_position_embeddings,
        d_model=model.config.hidden_size,
        layer_path="model.layers",
        replacement_class=GemmaReplacement,
    )
    model.eval()
    model.requires_grad_(False)

print(model)
print(mtm.memory_max)
print(mtm.memory_cur)

/cloud-dev/.venv/lib/python3.13/site-packages/codefind/registry.py:46: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, types.FunctionType):


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

GemmaReplacementInstance(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemm

In [45]:
from concurrent.futures import ThreadPoolExecutor

from transformers_sae.ops import find_latest_checkpoint, load_checkpoint

NUM_TRAINING_TOKENS = int(1e8)

def load_saes(checkpoint_dir: str):
    saes = {}

    def load_layer_checkpoint(layer):
        checkpoint = find_latest_checkpoint(checkpoint_dir, layer)
        if checkpoint is not None:
            cp = load_checkpoint(checkpoint)
            if cp.total_tokens_trained < NUM_TRAINING_TOKENS:
                print(f"No checkpoint found for layer {layer}")
                return layer, None
            sae = load_checkpoint(checkpoint).sae
            sae.eval()
            sae.onload()
            print(f"Loaded checkpoint for layer {layer}")
            return layer, sae
        else:
            print(f"No checkpoint found for layer {layer}")
            return layer, None

    # Load the latest checkpoints for each layer in parallel
    with ThreadPoolExecutor() as executor:
        results = executor.map(
            load_layer_checkpoint, range(model.num_layers - 1, -1, -1)
        )
        for layer, sae in results:
            if sae is not None:
                saes[layer] = sae

    return saes

# saes = load_saes("/workspace/sae_checkpoints/gemma_2_2b/next_layer_finetuned_interaction")
saes = load_saes("/workspace/sae_checkpoints/gemma_2_2b/next_layer_finetuned")

Loaded checkpoint for layer 1
Loaded checkpoint for layer 3
Loaded checkpoint for layer 4
Loaded checkpoint for layer 2
Loaded checkpoint for layer 5
Loaded checkpoint for layer 10
Loaded checkpoint for layer 17
Loaded checkpoint for layer 20
Loaded checkpoint for layer 25
Loaded checkpoint for layer 12
Loaded checkpoint for layer 21
Loaded checkpoint for layer 6
Loaded checkpoint for layer 22
Loaded checkpoint for layer 13
Loaded checkpoint for layer 24
Loaded checkpoint for layer 15
Loaded checkpoint for layer 0
Loaded checkpoint for layer 7
Loaded checkpoint for layer 16
Loaded checkpoint for layer 11
Loaded checkpoint for layer 19
Loaded checkpoint for layer 23
Loaded checkpoint for layer 18
Loaded checkpoint for layer 14
Loaded checkpoint for layer 8
Loaded checkpoint for layer 9


In [47]:
import cloudpickle

replacement_thresholds = cloudpickle.load(
    open(
        # "/workspace/sae_checkpoints/validations/gemma_2_2b/next_layer_finetuned_interaction/0.activation_thresholds",
        "/workspace/sae_checkpoints/validations/gemma_2_2b/next_layer_finetuned/0.activation_thresholds",
        "rb",
    )
)
for layer, sae in saes.items():
    for i, a in enumerate(sae.encoder.activation):
        a.threshold.fill_(replacement_thresholds[layer][i])


In [48]:
from transformers_sae.ops import generate
from transformers_sae.replacement_model import make_replacement_model

with torch.inference_mode():
    generate(
        ["The capital of France,",],
        make_replacement_model(model, saes),
        tokenizer,
        stream=True,
    )


 Paris, is a place that that that that the first time to be able and the and and and


In [49]:
from transformers_sae.benchmark import BenchmarkModel, MMLUBenchmark, MMLUTask

# Somewhat arbitrary list of tasks; these are the first 10 from the DeepEval enum that
# result in tokenizations that are short enough for our SAE's context window.
tasks = [
    MMLUTask.BUSINESS_ETHICS,
    MMLUTask.CLINICAL_KNOWLEDGE,
    MMLUTask.MEDICAL_GENETICS,
    MMLUTask.HIGH_SCHOOL_PHYSICS,
    MMLUTask.VIROLOGY,
    MMLUTask.HIGH_SCHOOL_MICROECONOMICS,
    MMLUTask.ECONOMETRICS,
    MMLUTask.COLLEGE_COMPUTER_SCIENCE,
    MMLUTask.HIGH_SCHOOL_BIOLOGY,
    MMLUTask.ABSTRACT_ALGEBRA,
]

# mmlu_baseline = run_mmlu(model, tokenizer, 16, tasks=[MMLUTask.HIGH_SCHOOL_BIOLOGY])
baseline_benchmark_model = BenchmarkModel(model, tokenizer)
mmlu_baseline = MMLUBenchmark(
    tokenizer,
    model.context_length,
    tasks=tasks,
    # tasks=[MMLUTask.HIGH_SCHOOL_BIOLOGY, MMLUTask.COLLEGE_COMPUTER_SCIENCE],
    # n_problems_per_task=16,
)
mmlu_baseline.evaluate(model=baseline_benchmark_model, batch_size=16);

Batch Processing business_ethics (batch_size=16):   0%|          | 0/7 [00:00<?, ?it/s]

MMLU Task Accuracy (task=business_ethics): 0.51


Batch Processing clinical_knowledge (batch_size=16):   0%|          | 0/17 [00:00<?, ?it/s]

MMLU Task Accuracy (task=clinical_knowledge): 0.5660377358490566


Batch Processing medical_genetics (batch_size=16):   0%|          | 0/7 [00:00<?, ?it/s]

...

MMLU Task Accuracy (task=college_computer_science): 0.41


Batch Processing high_school_biology (batch_size=16):   0%|          | 0/20 [00:00<?, ?it/s]

MMLU Task Accuracy (task=high_school_biology): 0.6387096774193548


Batch Processing abstract_algebra (batch_size=16):   0%|          | 0/7 [00:00<?, ?it/s]

MMLU Task Accuracy (task=abstract_algebra): 0.26
Overall MMLU Accuracy: 0.49939172749391725
Mean probability for any multiple choice answer:  0.9945137047564606
Mean probability for correct answer:  0.40317068933557443
Mean probability for correct answer, conditional on giving a multiple choice answer:  0.40524564339609487
Distribution of top logit answers:  Counter({' D': 494, ' C': 487, ' B': 390, ' A': 273})
Distribution of correct answers:  Counter({'D': 468, 'B': 413, 'C': 389, 'A': 374})


In [50]:
from transformers_sae.benchmark import BenchmarkModel, MMLUBenchmark, MMLUTask

sae_benchmark_model = BenchmarkModel(make_replacement_model(model, saes), tokenizer)
mmlu_sae_model = MMLUBenchmark(
    tokenizer,
    model.context_length,
    tasks=tasks,
)
mmlu_sae_model.evaluate(model=sae_benchmark_model, batch_size=16);

Batch Processing business_ethics (batch_size=16):   0%|          | 0/7 [00:00<?, ?it/s]

MMLU Task Accuracy (task=business_ethics): 0.0


Batch Processing clinical_knowledge (batch_size=16):   0%|          | 0/17 [00:00<?, ?it/s]

MMLU Task Accuracy (task=clinical_knowledge): 0.0


Batch Processing medical_genetics (batch_size=16):   0%|          | 0/7 [00:00<?, ?it/s]

...

MMLU Task Accuracy (task=college_computer_science): 0.01


Batch Processing high_school_biology (batch_size=16):   0%|          | 0/20 [00:00<?, ?it/s]

MMLU Task Accuracy (task=high_school_biology): 0.0


Batch Processing abstract_algebra (batch_size=16):   0%|          | 0/7 [00:00<?, ?it/s]

MMLU Task Accuracy (task=abstract_algebra): 0.0
Overall MMLU Accuracy: 0.0006082725060827251
Mean probability for any multiple choice answer:  0.002175223356763044
Mean probability for correct answer:  0.0005222365035188663
Mean probability for correct answer, conditional on giving a multiple choice answer:  0.23546554470177825
Distribution of top logit answers:  Counter({' ': 904, ' True': 114, ' T': 94, ' (': 43, ' m': 34, ' their': 31, '\n': 24, ' the': 21, '\n\n': 20, ' only': 17, ' c': 15, ' in': 15, ' one': 13, ' all': 11, ' at': 10, ' total': 10, ' a': 8, ' build': 8, ' as': 8, '1': 7, ' more': 7, ' into': 6, ' type': 6, ' b': 6, ' non': 6, ' no': 6, ' v': 6, ' down': 5, ' first': 5, ' to': 5, '.': 5, ' not': 4, ' part': 4, ' these': 4, ' current': 4, ' could': 4, ' cell': 4, '3': 3, ' best': 3, ' over': 3, ' social': 3, ' about': 3, ' end': 3, ' second': 3, ' central': 3, ' test': 3, ' n': 3, ' |': 3, ' work': 2, ' business': 2, ' hand': 2, ' back': 2, ' after': 2, ' every': 2,

In [ ]:
import cloudpickle
cloudpickle.dump(mmlu_baseline, open("/workspace/sae_checkpoints/benchmarks/baseline", "wb"))

In [16]:
import cloudpickle
cloudpickle.dump(mmlu_sae, open("/workspace/sae_checkpoints/benchmarks/next_layer_finetuned_interaction", "wb"))

In [29]:
from collections import Counter
print(Counter(mmlu_sae.predictions["Prediction"]))
print(Counter(mmlu_baseline.predictions["Prediction"]))

Counter({'A': 284, 'B': 19, 'D': 5, 'C': 2})
Counter({'C': 91, 'D': 79, 'B': 78, 'A': 62})
